<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/16_ml_intro/16_4_Pandas_Series.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction To Pandas Series

> **How to use this notebook.** This is the longest notebook in the course, and it is built to be a **reference**, not a story. On your first pass, work through it at skimming speed: run the cells, get the shape of what a Series can do, and don't try to memorize the methods. The units that follow (17 and 18) will send you back here whenever you need a specific tool — selection with `.loc`/`.iloc`, boolean filtering, string methods — and *that* is when the details will stick. Depth on demand beats memorization up front.

---

## Learning Objectives

After working through this notebook, you should be able to:

- Explain how a Pandas Series differs from a Python list and a NumPy array
- Create Series from lists, with and without a custom index
- Distinguish a copy from an alias, and know when each is created
- Select data by label (`.loc[]`), by position (`.iloc[]`), and by condition (boolean masks)
- Update values and index labels, and persist changes by reassignment
- Sort a Series by index or by values
- Apply vectorized math and string operations (the `.str` accessor) to a whole Series

---

## Lists vs. Arrays vs. Series

We are now going to start working with a Python library called Pandas. Pandas is a library that allows us to work with lots of data in some pretty sophisticated ways. The first part of Pandas that we are going to learn is the Pandas Series object. Before we get into the details,let's compare and contrast Pandas series with Python lists and Numpy arrays.

---

Python lists, NumPy arrays, and Pandas Series are the three most common ways to store collections of data in Python. While they may look similar, they are designed for different purposes and have significantly different performance characteristics.

### **Quick Comparison Table**

| Feature | **Python List** | **NumPy Array** | **Pandas Series** |
| :--- | :--- | :--- | :--- |
| **Data Types** | Heterogeneous (mixed) | Homogeneous (same type) | Homogeneous (usually) |
| **Indexing** | Integer-based only | Integer-based & Multi-D | **Labeled Indexing** |
| **Performance** | Slower (loop-based) | Fastest (vectorized) | Fast (built on NumPy) |
| **Memory** | High (stores object refs) | Low (contiguous block) | Moderate (includes index) |
| **Dimensions** | 1D (nested for higher) | N-Dimensional | 1-Dimensional only |
| **Missing Data** | No special handling | `np.nan` (floats only) | **Native NaN handling** |

---

### **1. Python List: The Generalist**
The built-in `list` is Python's "jack-of-all-trades." It is highly flexible but not optimized for math.
*   **Heterogeneity:** You can store an integer, a string, and another list all in one place: `[1, "hello", [3, 4]]`.
*   **Memory Structure:** It stores **pointers** (references) to objects scattered in memory. This "pointer chasing" makes it slow for large-scale data processing.
*   **Operations:** To perform math, you must use explicit `for` loops or list comprehensions.
    *   *Example:* `[x + 1 for x in my_list]`
*   **Best For:** Small collections, general-purpose programming, and storing mixed data types.

### **2. NumPy Array: The Speed Demon**
NumPy is the foundation of scientific computing in Python. Its `ndarray` (n-dimensional array) is designed for raw performance.
*   **Homogeneity:** All elements must be the same type (e.g., all `float64`). If you mix types, NumPy will "upcast" them (converting everything to strings, for example).
*   **Vectorization:** It uses **Single Instruction, Multiple Data (SIMD)**, allowing it to perform math on an entire array at once without a loop.
    *   *Example:* `my_array + 1` (instantly adds 1 to every element).
*   **Memory Structure:** It stores data in a **contiguous block**, which is extremely cache-friendly for CPUs.
*   **Best For:** Heavy mathematical computations, image processing, and machine learning (most ML libraries require NumPy arrays as input).

### **3. Pandas Series: The Labeled Specialist**
A Series is essentially a **1D NumPy array with labels**. It is the primary building block of a Pandas DataFrame.
*   **Labeled Indexing:** Unlike lists or NumPy arrays (which use `0, 1, 2...`), a Series allows you to use custom labels (e.g., dates, names, or IDs).
    *   *Example:* Accessing data via `series['2023-01-01']`.
*   **Data Alignment:** When you add two Series, Pandas automatically aligns them by their labels, even if the labels are in a different order.
*   **Handling Missing Data:** Pandas has robust built-in support for "Not a Number" (`NaN`) values and provides functions like `.fillna()` or `.dropna()`.
*   **Best For:** Data analysis, time-series data, and "dirty" real-world datasets that need cleaning and labeling.

---

### **When to use which?**

1.  **Use a Python List** when you have a small number of items and the types are different (e.g., a list of configuration settings).
2.  **Use a NumPy Array** when you are doing "pure math" on millions of numbers and need the absolute maximum speed and lowest memory usage.
3.  **Use a Pandas Series** when you are doing data science. If your data comes from a CSV, has headers, or has missing values, the labeling and alignment features of Pandas will save you hours of manual coding.

---

We will explore a few of the things Series can do in this course, but there's also a lot we are not going to cover. If you get interested, [the Pandas documentation covers the wide variety of methods available for Series objects](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.html).


## Importing Pandas
By convention, Pandas is imported with the alias `pd`.

In [1]:
# imports
import pandas as pd

### Series and the Song Dataset

In the first notebook we worked with a song dataset that had columns for tempo, energy, and danceability. Here, we will treat those columns as Pandas **Series**, and see that a Series can do everything the NumPy arrays from the previous notebook could do — plus more, once we give it a labeled index.

In [2]:
# The columns of our song dataset are just Pandas Series
song_tempos = pd.Series([120, 95, 145, 130, 110], name='Tempo')
song_energies = pd.Series([0.8, 0.3, 0.9, 0.7, 0.5], name='Energy')
print('Tempos:')
print(song_tempos)
print()
print('Energies:')
print(song_energies)

# Series support the same stats methods we used on NumPy arrays
print()
print(f'Mean tempo: {song_tempos.mean()} BPM')
print(f'Max energy: {song_energies.max()}')

Tempos:
0    120
1     95
2    145
3    130
4    110
Name: Tempo, dtype: int64

Energies:
0    0.8
1    0.3
2    0.9
3    0.7
4    0.5
Name: Energy, dtype: float64

Mean tempo: 120.0 BPM
Max energy: 0.9


## Creating and Copying Series

### Creating Empty Series

To create a new series, we can use the `pd.Series` method. When we create the series, Pandas would like to know what kind of data we intend to store in the series. We could specify integers, `dtype='Int64'`, strings, `'dtype='string'`, and [other things](https://pandas.pydata.org/pandas-docs/stable/user_guide/basics.html#basics-dtypes). If we want to mix data types within the series, we can specify 'object' as the data type using the `dtype='object'` argument.


In [3]:
# examples of creating empty series objects
my_integer_series = pd.Series(dtype='int64')
my_string_series = pd.Series(dtype='string')
my_mixed_series = pd.Series(dtype='object')

# check the empty series objects
series_list = [my_integer_series, my_string_series, my_mixed_series]
for s in series_list:
  print(f'{s}, length = {len(s)} ')

Series([], dtype: int64), length = 0 
Series([], dtype: string), length = 0 
Series([], dtype: object), length = 0 


### Creating Series from Lists

More commonly, we won't start with empty series, instead we will create a series using existing data from some other kind of object. If we have the data in some form and want to put it into a series, we again use `pd.Series` but we do so using the data source as an argument. Below we create series from lists.

In [4]:
integer_list = [11, 12, 13, 14, 15]
string_list = ['apple', 'banana', 'cherry', 'daikon', 'eggplant']
mixed_list = integer_list + string_list

# examples of creating series objects from lists
my_integer_series = pd.Series(integer_list, dtype='int64')
my_string_series = pd.Series(string_list, dtype='string')
my_mixed_series = pd.Series(mixed_list, dtype='object')

# check the series objects
series_list = [my_integer_series, my_string_series, my_mixed_series]
for s in series_list:
  print(f'{s}, length = {len(s)} ')
  print()


0    11
1    12
2    13
3    14
4    15
dtype: int64, length = 5 

0       apple
1      banana
2      cherry
3      daikon
4    eggplant
dtype: string, length = 5 

0          11
1          12
2          13
3          14
4          15
5       apple
6      banana
7      cherry
8      daikon
9    eggplant
dtype: object, length = 10 



When we create a series from existing data we do not have to specify the datatype using the `dtype` argument. If we exclude the argument, Pandas will attempt to infer the dtype of the series. Often this automatic inference works as you'd like, but there are occasions where the data you are building the series from cause some complications. We will return to this issue in a bit when we discuss dataframes.  

In [5]:
# examples of creating series objects from lists without specifying the data type
my_integer_series = pd.Series(integer_list)
my_string_series = pd.Series(string_list)
my_mixed_series = pd.Series(mixed_list)

# check the series objects
series_list = [my_integer_series, my_string_series, my_mixed_series]
for s in series_list:
  print(f'{s}, length = {len(s)} ')
  print()

0    11
1    12
2    13
3    14
4    15
dtype: int64, length = 5 

0       apple
1      banana
2      cherry
3      daikon
4    eggplant
dtype: str, length = 5 

0          11
1          12
2          13
3          14
4          15
5       apple
6      banana
7      cherry
8      daikon
9    eggplant
dtype: object, length = 10 



### Creating a Series with an Index
So far we've created series that have specified values, but no specified index. When we do this, Pandas assigns a range object to the index that generates integer values for each row, and we wind up with something that has an index much like a list.

We can specify an index using an optional argument when we construct the series. In the code below,  we are using two lists to specify the values and the index.

In [6]:
fruit_name_list = ['apple', 'banana', 'cherry', 'dates', 'elderberry']
fruit_weight_list = [180, 120, 15, 650, 450]

my_fruit_series = pd.Series(data=fruit_weight_list,
                            index=fruit_name_list)
my_fruit_series

apple         180
banana        120
cherry         15
dates         650
elderberry    450
dtype: int64

### Copying Series
Making a copy of a series works much like copying a list: assignment creates an alias (a second name for the same object), while `.copy()` creates a new independent object. We'll demonstrate this with series below.

When working with series, avoid making a copy unless you have a specific reason to do so. Copies consume extra memory and can slow your program down — especially with large datasets. Most of the time you can work with the original series or an alias.

In [7]:
# create a new series
fruit_name_list = ['apple', 'banana', 'cherry', 'dates', 'elderberry']
fruit_weight_list = [180, 120, 15, 650, 450]

my_fruit_series = pd.Series(data=fruit_weight_list, index=fruit_name_list)

# this does not make a copy of the series, only an alias
alias_not_a_new_series = my_fruit_series

# this makes a copy of the series
new_series_not_an_alias = my_fruit_series.copy()

In [8]:
# they contain the same values, note: ignore how this code works for now
print(alias_not_a_new_series.equals(new_series_not_an_alias))

# but they are not the same object
print(alias_not_a_new_series is new_series_not_an_alias)

# while these two are the same object
print(alias_not_a_new_series is my_fruit_series)

True
False
True


What we will usually do instead is simply *select* the part of the series we want to see. The selection operations in this section — indexing, slicing, boolean masking — return a **new object containing the result and leave the original series unchanged**. If we want to keep a result, we assign it to a variable; otherwise it is displayed and discarded. The original series is never modified unless we explicitly assign the result back to it. This idea may feel a bit fuzzy for a while, but it will get better as we work through the exercises.

## Examining Series

We will often be working with series that contain a large number of values. If we have a series with 50000 values in it, and then we tell Python to print every value, we are going to have too much output to deal with. As a convenience, large series are usually displayed with values in the middle omitted.



In [9]:
# create a big list of random integers
import random
big_list = []
for v in range(50000):
  big_list.append(random.randint(1,100))

# convert the big list to a big series
big_series = pd.Series(data = big_list, dtype='int64')
len(big_series)

50000

In [10]:
# examine the big series; notice the truncation of the middle
big_series

0        98
1        45
2        46
3        43
4        63
         ..
49995    43
49996    79
49997    74
49998    92
49999    18
Length: 50000, dtype: int64

We can use the method `.head()` or `.tail()` to specifically look at the beginning or end of the series.

In [11]:
# using .head()
big_series.head()

0    98
1    45
2    46
3    43
4    63
dtype: int64

In [12]:
# using .tail()
big_series.tail()

49995    43
49996    79
49997    74
49998    92
49999    18
dtype: int64

By default, `.head()` and `.tail()` display five labels with their values. This can be modified using an optional argument.

In [13]:
# using .head() with an argument
big_series.head(10)

0    98
1    45
2    46
3    43
4    63
5    72
6    65
7    62
8    25
9    76
dtype: int64

In [14]:
# using .tail() with an argument
big_series.tail(3)

49997    74
49998    92
49999    18
dtype: int64

When working with data, we will often use `.head()` or `.tail()` to simply check that our code is working as intended, much like you would do with `print()` while writing your code.

## Selecting Data


### Selecting Data Using the Index
The lists we are used to working with have an index of integer values that start with 0 and increase by one for every value in the list. In contrast, a series has an index that can be anything we like. You can think of it as a second set of values that can be associated with the data in the series.

We will refer to the values in the index as labels. We can use the labels to retrieve subsets of data from our series.

In [15]:

# create lists of fruits and weights
fruit_name_list = ['apple', 'banana', 'cherry', 'dates', 'elderberry']
fruit_weight_list = [180, 120, 15, 650, 450]

# create series from lists
my_fruit_series = pd.Series(data=fruit_weight_list,
                            index=fruit_name_list)
my_fruit_series

apple         180
banana        120
cherry         15
dates         650
elderberry    450
dtype: int64

Notice that the datatype of the series is based on the values in the series, not on the values in the index.

Now that our series has both an index and values, we can access those values separately using dot notation.

In [16]:
# examine the series index attribute
my_fruit_series.index

Index(['apple', 'banana', 'cherry', 'dates', 'elderberry'], dtype='str')

In [17]:
# examine the series values attribute
my_fruit_series.values

array([180, 120,  15, 650, 450])

The values of a series index do not have to be unique.

In [18]:
# create lists for a series with repeated values in index
fruit_name_list = ['apple', 'apple', 'apple', 'banana', 'banana']
fruit_weight_list = [180, 120, 15, 650, 450]

# constructs series with repeated values in the index
my_fruit_series = pd.Series(data=fruit_weight_list,
                            index=fruit_name_list)
my_fruit_series

apple     180
apple     120
apple      15
banana    650
banana    450
dtype: int64

The values in an index also do not need to be sequential, and can be of any data type.

In [19]:
# create lists for constructing a series with a non-sequential index
fruit_name_list = ['banana', 'banana', 'apple', 'apple', 'apple']
fruit_weight_list = [180, 120, 15, 650, 450]

# constructs a series with a non-sequential index
my_fruit_series = pd.Series(data=fruit_name_list,
                            index=fruit_weight_list)
my_fruit_series

180    banana
120    banana
15      apple
650     apple
450     apple
dtype: str

### Selection by Label Using `.loc[]`
To get or update values in a series, we would use square brackets in much the same way we would with lists. However, we are going add the method `.loc` before the brackets to specify that we are indexing or slicing based on the labels in the index.


In [20]:
fruit_name_list = ['apple', 'banana', 'cherry', 'dates', 'elderberry']
fruit_weight_list = [180, 120, 15, 650, 450]

my_fruit_series = pd.Series(data=fruit_weight_list,
                            index=fruit_name_list)
my_fruit_series

apple         180
banana        120
cherry         15
dates         650
elderberry    450
dtype: int64

In [21]:
# indexing a series values using .loc and an index label
my_fruit_series.loc['apple']

np.int64(180)

In [22]:
# indexing a series values using .loc and an index label
my_fruit_series.loc['dates']

np.int64(650)

We can also use label based slicing to get a range of values.

In [23]:
# note the label and value of cherry are included
my_fruit_series.loc['apple':'cherry']

apple     180
banana    120
cherry     15
dtype: int64

Note: Unlike normal Python slicing, which would usually go up to, but not include, the stop value, slicing with `.loc` includes the 'stop' value.

In [24]:
# slicing from a label until the end of the series
my_fruit_series.loc['cherry':]

cherry         15
dates         650
elderberry    450
dtype: int64

Since multiple values in the series can have the same 'label' or value for the index, indexing will return all values with a label, rather than just one.

In [25]:
# constructs a series with repeated values in index
fruit_name_list = ['apple', 'apple', 'apple', 'banana', 'banana']
fruit_weight_list = [180, 120, 15, 650, 450]
my_fruit_series = pd.Series(data=fruit_weight_list,
                            index=fruit_name_list)

# indexing by index label when there are repeated values in index
my_fruit_series.loc['apple']

apple    180
apple    120
apple     15
dtype: int64

### Selection by Position Using `.iloc[]`

If you want to ignore the labels in the index and select values based on position, in the same way you do with a Python list, you can use 'implicit' indexing with `.iloc`.

Note: unlike `.loc`, slicing with `.iloc` works just like it does in python lists and strings; the stop value is not included.

In [26]:
fruit_name_list = ['apple', 'banana', 'cherry', 'dates', 'elderberry']
fruit_weight_list = [180, 120, 15, 650, 450]

my_fruit_series = pd.Series(data=fruit_weight_list,
                            index=fruit_name_list)

my_fruit_series.iloc[1:3]

banana    120
cherry     15
dtype: int64

We can't do negative indexing using `.loc`, since it's looking for labels, but we can do negative indexing with `.iloc`.

In [27]:
fruit_name_list = ['apple', 'banana', 'cherry', 'dates', 'elderberry']
fruit_weight_list = [180, 120, 15, 650, 450]

my_fruit_series = pd.Series(data=fruit_weight_list,
                            index=fruit_name_list)

my_fruit_series.iloc[-4:-1]

banana    120
cherry     15
dates     650
dtype: int64

We are going to spend a lot of time in this class asking Pandas to look at a series and identify some subset of values that are associated with a particular label or set of labels, so `.loc` will usually suffice. We will rarely use implicit indexing, so we are not going to practice it. However, it's important you know that it exists and that you recognize that when we use `.loc` we are relying on labels in the index rather than positions.



### Selection by Condition Using Booleans

When we use logical operators with a series, we get back a series full of Boolean values. Each value in the result is `True` if that element satisfied the condition, and `False` if it did not. The code cells below show a few examples.

In [28]:
fruit_name_list = ['apple', 'banana', 'cherry', 'dates', 'elderberry']
fruit_weight_list = [180, 120, 15, 45, 75]  # note: new, lighter weights than earlier examples

my_fruit_series = pd.Series(data=fruit_weight_list,
                            index=fruit_name_list)

my_fruit_series < 100

apple         False
banana        False
cherry         True
dates          True
elderberry     True
dtype: bool

In [29]:
my_fruit_series == 45

apple         False
banana        False
cherry        False
dates          True
elderberry    False
dtype: bool

In [30]:
my_fruit_series >= 100

apple          True
banana         True
cherry        False
dates         False
elderberry    False
dtype: bool

These series of booleans can be used as a mask to select specific values that meet a condition. We can do this by using the mask as an index.

In [31]:
# create the boolean mask and assign to a variable
heavy_fruit_mask = (my_fruit_series >= 100)

# use the mask variable to index into the series
my_fruit_series.loc[heavy_fruit_mask]

apple     180
banana    120
dtype: int64

We will be doing a lot of Boolean masking in this course. At some points, we will be combining three or four Boolean masks to select some particular subset of data. All of this can be done in a single line of code, but I am going to ask you to do it in multiple steps to help with troubleshooting your code.

In [32]:
# boolean masking done in a single step
my_fruit_series.loc[my_fruit_series <= 100]

cherry        15
dates         45
elderberry    75
dtype: int64

In [33]:
# the same thing done in two steps

# step 1: create the boolean mask and assign to a variable
light_fruit_mask = (my_fruit_series <= 100)

# step2: use the mask variable to index into the series
my_fruit_series.loc[light_fruit_mask]

cherry        15
dates         45
elderberry    75
dtype: int64

In [34]:
# an example using multiple boolean masks

# step 1: create the boolean mask and assign to a variable
light_fruit_mask = (my_fruit_series <= 20)

# step 2: create the boolean mask and assign to a variable
heavy_fruit_mask = (my_fruit_series >= 100)

# step 3: use the mask variables to index into the series
# the '|' substitutes for 'or'. Use | and & with masks (not 'and'/'or'),
# because Pandas evaluates element-wise, not the single True/False Python expects.
my_fruit_series.loc[light_fruit_mask | heavy_fruit_mask]

apple     180
banana    120
cherry     15
dtype: int64

## Series: Updating and Sorting

### Updating Values
Like a list, a series is mutable, so values can be updated using indexing with `.loc` or `iloc`.

In [35]:
# import pandas

# create fruit weight series from lists
fruit_name_list = ['apple', 'banana', 'cherry', 'dates', 'elderberry']
fruit_weight_list = [180, 120, 15, 45, 75]
my_fruit_series = pd.Series(data=fruit_weight_list, index=fruit_name_list)

# update a value based on a label
my_fruit_series.loc['cherry'] = 25
my_fruit_series

apple         180
banana        120
cherry         25
dates          45
elderberry     75
dtype: int64

In [36]:
# update a value based on a position
my_fruit_series.iloc[2] = 225
my_fruit_series

apple         180
banana        120
cherry        225
dates          45
elderberry     75
dtype: int64

You can also update multiple values simultaneously.

In [37]:
# update multiple values based on label
my_fruit_series.loc['apple':'cherry'] = [200, 150, 35]
my_fruit_series

apple         200
banana        150
cherry         35
dates          45
elderberry     75
dtype: int64

In [38]:
# update multiple values based on position
my_fruit_series.iloc[2:] = [300, 400, 500]
my_fruit_series

apple         200
banana        150
cherry        300
dates         400
elderberry    500
dtype: int64

Since multiple values can share the same label in the index, multiple values that share a value can be updated simultaneously.

In [39]:
# constructs a series with repeated values in index
fruit_name_list = ['apple', 'apple', 'apple', 'banana', 'banana']
fruit_weight_list = [180, 120, 15, 650, 450]

my_fruit_series = pd.Series(data = fruit_weight_list, index = fruit_name_list)
my_fruit_series

apple     180
apple     120
apple      15
banana    650
banana    450
dtype: int64

In [40]:
# updating multiple values with a single assignment
my_fruit_series.loc['apple'] = 333
my_fruit_series

apple     333
apple     333
apple     333
banana    650
banana    450
dtype: int64

Note: setting or 'updating' variables with `.loc` or `.iloc`, as we have in this section, changes the series directly. We do not need to make a copy of the series to make the change.

### Updating Index Labels
The series index object is not mutable, so we cannot use positional indexing to update a single label within the index. However, we can reassign a list of new values to the index.

In [41]:
my_fruit_series.index

Index(['apple', 'apple', 'apple', 'banana', 'banana'], dtype='str')

In [42]:
my_fruit_series.index = ['apple', 'banana', 'cherry', 'dates', 'elderberry']
my_fruit_series.index

Index(['apple', 'banana', 'cherry', 'dates', 'elderberry'], dtype='str')

## Basic Series Operations
We can operate on all the values in a series at once without using a for loop. We have some examples below using mathematical operators and string operations.

### Basic Math Operations
If we had a list full of integers and we were interested in adding five to each value in the list, we would need to write a for loop to loop over the list and update each value. This sort of operation is much simpler with a series.

In [43]:
my_fruit_series

apple         333
banana        333
cherry        333
dates         650
elderberry    450
dtype: int64

In [44]:
my_fruit_series + 5

apple         338
banana        338
cherry        338
dates         655
elderberry    455
dtype: int64

In [45]:
my_fruit_series - 5

apple         328
banana        328
cherry        328
dates         645
elderberry    445
dtype: int64

In [46]:
my_fruit_series * 5

apple         1665
banana        1665
cherry        1665
dates         3250
elderberry    2250
dtype: int64

In [47]:
my_fruit_series / 5

apple          66.6
banana         66.6
cherry         66.6
dates         130.0
elderberry     90.0
dtype: float64

Important note! In all the previous examples of operations, we have not actually made any change to the underlying series. Each operation returns a **new Series** as its result, but that result is discarded unless we assign it to a variable. If we want to make a lasting alteration of the series, we have to perform the operation and assign the result to a variable.

Also note: dividing an integer series by a scalar produces a `float64` result (as seen above with `/ 5`). Pandas automatically promotes the dtype when the operation can produce fractional values. This is expected behavior.

In [48]:
# this creates a new series with five added to each value
# and reassigns the result to the original variable name
my_fruit_series = my_fruit_series + 5
my_fruit_series

apple         338
banana        338
cherry        338
dates         655
elderberry    455
dtype: int64

### Basic String Operations

In [49]:
# creates a series with strings as values
fruit_name_list = ['banana', 'banana', 'apple', 'apple', 'apple']
fruit_weight_list = [180, 120, 15, 650, 450]

my_fruit_series = pd.Series(data=fruit_name_list, index=fruit_weight_list)
my_fruit_series

180    banana
120    banana
15      apple
650     apple
450     apple
dtype: str

In [50]:
# creating a new series using string concatenation
my_fruit_series + ' is a fruit!'

180    banana is a fruit!
120    banana is a fruit!
15      apple is a fruit!
650     apple is a fruit!
450     apple is a fruit!
dtype: str

As we saw in the previous section with the basic operations, performing the operations does not change the underlying series. If we want to retain the change we have to reassign the result over the original.

In [51]:
# this doesn't change anything
my_fruit_series + ' is a fruit!'
my_fruit_series

180    banana
120    banana
15      apple
650     apple
450     apple
dtype: str

In [52]:
# but this does
my_fruit_series = my_fruit_series + ' is a fruit!'
my_fruit_series

180    banana is a fruit!
120    banana is a fruit!
15      apple is a fruit!
650     apple is a fruit!
450     apple is a fruit!
dtype: str

## Sorting Series by Index and Value
When we have wanted to sort a Python list, we have relied on the `.sort()` list method which works by sorting the list 'in place' and returning None. Sorting a series works much differently, for a few reasons. First, when we are working with a series, we can sort on either the index or sort on the values.

To sort on the index, we use `.sort_index()`

In [53]:
# sort on the index
my_fruit_series.sort_index()

15      apple is a fruit!
120    banana is a fruit!
180    banana is a fruit!
450     apple is a fruit!
650     apple is a fruit!
dtype: str

To sort on the values, we use `.sort_values()`.

In [54]:
# sorting on the values
my_fruit_series.sort_values()

15      apple is a fruit!
650     apple is a fruit!
450     apple is a fruit!
180    banana is a fruit!
120    banana is a fruit!
dtype: str

Unlike sorting a list, when you sort a series, the sorting is not done 'in place'. The previous two code cells show the series sorted by index and then sorted by values, but in both cases the result was a new Series object that we did not assign to a variable, so the original series remains in its original order.

In [55]:
# the actual series object was not changed
my_fruit_series

180    banana is a fruit!
120    banana is a fruit!
15      apple is a fruit!
650     apple is a fruit!
450     apple is a fruit!
dtype: str

If we want to retain the sorted series, we can reassign the result over the original series.

In [56]:
my_fruit_series = my_fruit_series.sort_values()
my_fruit_series

15      apple is a fruit!
650     apple is a fruit!
450     apple is a fruit!
180    banana is a fruit!
120    banana is a fruit!
dtype: str

In [57]:
my_fruit_series = my_fruit_series.sort_index()
my_fruit_series

15      apple is a fruit!
120    banana is a fruit!
180    banana is a fruit!
450     apple is a fruit!
650     apple is a fruit!
dtype: str

Pandas does allow for in-place sorting using `inplace=True`. However, mixing the in-place style with the reassignment style we use everywhere else is error-prone and makes code harder to reason about — the pandas developers themselves discourage `inplace=True`. To keep things simple, do not use `inplace=True` this semester.

Similar to `.sort()`, both `.sort_index()` and `.sort_values()` have optional arguments to control the direction of the sorting.

In [58]:
my_fruit_series = my_fruit_series.sort_index(ascending=False)
my_fruit_series

650     apple is a fruit!
450     apple is a fruit!
180    banana is a fruit!
120    banana is a fruit!
15      apple is a fruit!
dtype: str

## Examples


I've grabbed some data on the estimated population of each U.S. State in 2019 [from this website](https://www.infoplease.com/us/states/state-population-by-rank).

We are going to put this data into a series and then work with it a bit using the tools we have introduced in this reading.


### Creating a Series

In [59]:
# import pandas

# lists of state names and populations
state_list = ['California', 'Texas', 'Florida', 'New York', 'Illinois',
              'Pennsylvania', 'Ohio', 'Georgia', 'North Carolina', 'Michigan',
              'New Jersey', 'Virginia', 'Washington', 'Arizona', 'Massachusetts',
              'Tennessee', 'Indiana', 'Missouri', 'Maryland', 'Wisconsin',
              'Colorado', 'Minnesota', 'South Carolina', 'Alabama', 'Louisiana',
              'Kentucky', 'Oregon', 'Oklahoma', 'Connecticut', 'Utah', 'Iowa',
              'Nevada', 'Arkansas', 'Mississippi', 'Kansas', 'New Mexico',
              'Nebraska', 'West Virginia', 'Idaho', 'Hawaii', 'New Hampshire',
              'Maine', 'Montana', 'Rhode Island', 'Delaware', 'South Dakota',
              'North Dakota', 'Alaska', 'DC', 'Vermont', 'Wyoming']

population_list = [39512223, 28995881, 21477737, 19453561, 12671821, 12801989,
                   11689100, 10617423, 10488084, 9986857, 8882190, 8535519,
                   7614893, 7278717, 6949503, 6833174, 6732219, 6137428,
                   6045680, 5822434, 5758736, 5639632, 5148714, 4903185,
                   4648794, 4467673, 4217737, 3956971, 3565287, 3205958,
                   3155070, 3080156, 3017825, 2976149, 2913314, 2096829,
                   1934408, 1792147, 1787065, 1415872, 1359711, 1344212,
                   1068778, 1059361, 973764, 884659, 762062, 731545, 705749,
                   623989, 578759,]

# create a series from a list of values and a list of labels
state_series = pd.Series(data=population_list,
                         index=state_list,
                         dtype='int64')

type(state_series)

pandas.Series

### Examining a Series

In [60]:
# examine with head
state_series.head()

California    39512223
Texas         28995881
Florida       21477737
New York      19453561
Illinois      12671821
dtype: int64

In [61]:
# examine with tail
state_series.tail()

North Dakota    762062
Alaska          731545
DC              705749
Vermont         623989
Wyoming         578759
dtype: int64

In [62]:
# examine values
state_series.values

array([39512223, 28995881, 21477737, 19453561, 12671821, 12801989,
       11689100, 10617423, 10488084,  9986857,  8882190,  8535519,
        7614893,  7278717,  6949503,  6833174,  6732219,  6137428,
        6045680,  5822434,  5758736,  5639632,  5148714,  4903185,
        4648794,  4467673,  4217737,  3956971,  3565287,  3205958,
        3155070,  3080156,  3017825,  2976149,  2913314,  2096829,
        1934408,  1792147,  1787065,  1415872,  1359711,  1344212,
        1068778,  1059361,   973764,   884659,   762062,   731545,
         705749,   623989,   578759])

In [63]:
# examine index
state_series.index

Index(['California', 'Texas', 'Florida', 'New York', 'Illinois',
       'Pennsylvania', 'Ohio', 'Georgia', 'North Carolina', 'Michigan',
       'New Jersey', 'Virginia', 'Washington', 'Arizona', 'Massachusetts',
       'Tennessee', 'Indiana', 'Missouri', 'Maryland', 'Wisconsin', 'Colorado',
       'Minnesota', 'South Carolina', 'Alabama', 'Louisiana', 'Kentucky',
       'Oregon', 'Oklahoma', 'Connecticut', 'Utah', 'Iowa', 'Nevada',
       'Arkansas', 'Mississippi', 'Kansas', 'New Mexico', 'Nebraska',
       'West Virginia', 'Idaho', 'Hawaii', 'New Hampshire', 'Maine', 'Montana',
       'Rhode Island', 'Delaware', 'South Dakota', 'North Dakota', 'Alaska',
       'DC', 'Vermont', 'Wyoming'],
      dtype='str')

### Selecting Data in a Series

In [64]:
# indexing by label
state_series.loc['Hawaii']

np.int64(1415872)

In [65]:
# slicing by label, note! includes stop value
state_series.loc['Illinois':'Indiana']

Illinois          12671821
Pennsylvania      12801989
Ohio              11689100
Georgia           10617423
North Carolina    10488084
Michigan           9986857
New Jersey         8882190
Virginia           8535519
Washington         7614893
Arizona            7278717
Massachusetts      6949503
Tennessee          6833174
Indiana            6732219
dtype: int64

In [66]:
# indexing by location
state_series.iloc[4]

np.int64(12671821)

In [67]:
# slicing by location, note! does not include stop value
state_series.iloc[1:4]

Texas       28995881
Florida     21477737
New York    19453561
dtype: int64

### Sorting a Series

In [68]:
# sort by index
state_series.sort_index().head()

Alabama        4903185
Alaska          731545
Arizona        7278717
Arkansas       3017825
California    39512223
dtype: int64

In [69]:
# sort by values
state_series.sort_values().head()

Wyoming         578759
Vermont         623989
DC              705749
Alaska          731545
North Dakota    762062
dtype: int64

### Basic Operations with a Series

In [70]:
# use .sum() to get the U.S. total population
total_population = state_series.sum()
total_population

np.int64(328300544)

In [71]:
# create a new series with the percent of pop. for each state
state_series_percent = state_series / total_population
state_series_percent.head()

California    0.120354
Texas         0.088321
Florida       0.065421
New York      0.059255
Illinois      0.038598
dtype: float64

In [72]:
# convert from decimal to percent and round
state_series_percent = state_series_percent * 100
state_series_percent = state_series_percent.round(2)
state_series_percent.head()

California    12.04
Texas          8.83
Florida        6.54
New York       5.93
Illinois       3.86
dtype: float64

### Small Program with Series

This program uses a Series method we haven't introduced yet:

- **`.cumsum()`** (cumulative sum) — returns a new Series where each value is the running total up to that row. For example, `pd.Series([10, 20, 30]).cumsum()` gives `[10, 30, 60]`. Here we use it to find the point where the accumulated population percentage crosses 50%.

In [73]:
# a little program to find the number of large states that account
# for more than 50% of the US population

# sort the values in the series to make sure largest states are at the top
state_series_percent = state_series_percent.sort_values(ascending=False)

# Use cumsum to find the running total without a manual for-loop
cumulative_percent = state_series_percent.cumsum()

# Find how many states it takes to exceed 50%
# (cumulative_percent < 50) gives a boolean series; .sum() counts the Trues;
# +1 because we need the state that pushes us over 50
num_states_to_50 = (cumulative_percent < 50).sum() + 1

# Get the names of those states
state_50percent_list = state_series_percent.head(num_states_to_50).index

print(f'The top {len(state_50percent_list)} U.S. States by population account '
      f'for more than 50% of the U.S. population.')
print('\nThese states include:\n')
for state in state_50percent_list:
  print(f'{state} with a population of {state_series[state]}.')

The top 9 U.S. States by population account for more than 50% of the U.S. population.

These states include:

California with a population of 39512223.
Texas with a population of 28995881.
Florida with a population of 21477737.
New York with a population of 19453561.
Pennsylvania with a population of 12801989.
Illinois with a population of 12671821.
Ohio with a population of 11689100.
Georgia with a population of 10617423.
North Carolina with a population of 10488084.


## String Methods

Hopefully you're feeling somewhat confident with the basics of series objects. The next bit we are going to tackle involves different methods of working the strings in a series.

Here's a series that contains information about the best-selling books of all time taken from [this article on wikipedia](https://en.wikipedia.org/wiki/List_of_best-selling_books).

In [74]:
# define file address
url = 'https://raw.githubusercontent.com/bsheese/cs377/main/data/data_book_bestsellers.csv'

# create series
df = pd.read_csv(url, header=0, names=['i', 'book'])
book_series = df.loc[:, 'book']

We have a series full of strings. In this case, each value in the series is a string with the title and author of a book. In a previous section, we saw how we can do simple string concatenation with a whole series at the same time.

In [75]:
book_exclaim = book_series + "!"
book_exclaim.head()

0    Harry Potter and the Philosopher's Stone by J....
1       The Little Prince by Antoine de Saint-Exupéry!
2              Dream of the Red Chamber by Cao Xueqin!
3                      The Hobbit by J. R. R. Tolkien!
4         And Then There Were None by Agatha Christie!
Name: book, dtype: str

You might be tempted to write a for loop to do what we've just done, and it's possible, we could iterate over the series and get the same result...

In [76]:
# iterating through a series with a for loop, try not to do this
book_exclaim_alt = []
for book in book_series:
  book_exclaim_alt.append(book + '!')
book_exclaim_alt = pd.Series(book_exclaim_alt, dtype='string')

# check the result
book_exclaim_alt.head()

0    Harry Potter and the Philosopher's Stone by J....
1       The Little Prince by Antoine de Saint-Exupéry!
2              Dream of the Red Chamber by Cao Xueqin!
3                      The Hobbit by J. R. R. Tolkien!
4         And Then There Were None by Agatha Christie!
dtype: string

As a general rule, if we can operate on a series without using a for-loop, we should. The reason is that Series operations are executed in compiled C code under the hood (via NumPy), while a Python for-loop interprets each iteration one at a time through the Python runtime. On a series with a million rows, the vectorized operation can be 100x faster than the equivalent loop. This performance gap is one of the main reasons to use Pandas over plain Python lists. For now, just remember that you often can do without for-loops when working with series (and with dataframes).

### Using the String Accessor
Many of the string methods we are used to working with, such as `.lower()`, `.upper()`, `.split()`, and `.replace()` can be applied to an entire series of string values. But to do so, we need to use the `.str` string accessor as follows:

In [77]:
book_series.str.lower().head()

0    harry potter and the philosopher's stone by j....
1        the little prince by antoine de saint-exupéry
2               dream of the red chamber by cao xueqin
3                       the hobbit by j. r. r. tolkien
4          and then there were none by agatha christie
Name: book, dtype: str

In [78]:
book_series.str.upper().head()

0    HARRY POTTER AND THE PHILOSOPHER'S STONE BY J....
1        THE LITTLE PRINCE BY ANTOINE DE SAINT-EXUPÉRY
2               DREAM OF THE RED CHAMBER BY CAO XUEQIN
3                       THE HOBBIT BY J. R. R. TOLKIEN
4          AND THEN THERE WERE NONE BY AGATHA CHRISTIE
Name: book, dtype: str

In [79]:
book_series.str.replace('Harry Potter', 'Hermione Granger').head()

0    Hermione Granger and the Philosopher's Stone b...
1        The Little Prince by Antoine de Saint-Exupéry
2               Dream of the Red Chamber by Cao Xueqin
3                       The Hobbit by J. R. R. Tolkien
4          And Then There Were None by Agatha Christie
Name: book, dtype: str

As a general rule, if there's a [Python string method](https://docs.python.org/3/library/stdtypes.html#string-methods) you want to use with a series, you can likely find [a Pandas equivalent](https://pandas.pydata.org/pandas-docs/version/0.23.4/text.html#).

### Length of String

In [80]:
# evaluates to a series of string lengths
book_series.str.len().head()

0    57
1    45
2    38
3    30
4    43
Name: book, dtype: int64

### Starts and Ends With

In [81]:
# evaluates to a series of booleans
book_series.str.startswith('Harry').head()

0     True
1    False
2    False
3    False
4    False
Name: book, dtype: bool

In [82]:
# evaluates to a series of booleans
book_series.str.endswith('Tolkien').head()

0    False
1    False
2    False
3     True
4    False
Name: book, dtype: bool

### Contains Sub-String

In [83]:
# evaluates to a series of booleans
book_series.str.contains('Agatha').head()

0    False
1    False
2    False
3    False
4     True
Name: book, dtype: bool

### Find Sub-String

In [84]:
# evaluates to a series of integers,
# -1 if substring not found, otherwise index of first occurrence
book_series.str.find('Christi').head()

0    -1
1    -1
2    -1
3    -1
4    35
Name: book, dtype: int64

### Split on Sub-String

In [85]:
# evaluates to series where each value contains a list
book_series.str.split(' by ').head()

0    [Harry Potter and the Philosopher's Stone, J. ...
1        [The Little Prince, Antoine de Saint-Exupéry]
2               [Dream of the Red Chamber, Cao Xueqin]
3                       [The Hobbit, J. R. R. Tolkien]
4          [And Then There Were None, Agatha Christie]
Name: book, dtype: object

In [86]:
# split each string into a list, return a series of lists with two elements
book_series_split = book_series.str.split(' by ')

# use the string accessor with indexing to get a series
# containing the first value in the list
book_titles = book_series_split.str[0]

# use the string accessor with indexing to get a series
# containing the second value in the list
book_authors = book_series_split.str[1]

#check result
book_authors.tail()

52        Daphne du Maurier
53            George Orwell
54    William Bradford Huie
55            Stieg Larsson
56                Dan Brown
Name: book, dtype: object

## Your Turn

1. Create a Pandas Series named `scores` from `[72, 88, 55, 94, 61]` with index labels `['Alice', 'Bob', 'Carol', 'Dave', 'Eve']`.
   - Use boolean masking to select scores above 80.
   - Use boolean masking to select scores that are either below 60 **or** above 90. You will need `|`.
   - Sort by value descending and use `.head(3)` to show the top 3 scorers.

2. Using the state population data from earlier in this notebook, write code that:
   - Creates a boolean mask for states with populations between 5 million and 15 million (inclusive). You will need two masks combined with `&`.
   - Prints the number of matching states and their names.

3. Create a Series of at least five movie titles (strings) with integer labels for the index. Use the `.str` accessor to:
   - Convert all titles to uppercase.
   - Find the character length of each title.
   - Find which titles contain the word `'the'` (case-insensitive). Hint: convert to lowercase first, then use `.str.contains()`.

4. **Challenge:** The state population series has state names as its index. Using `.str.contains(' ')`, identify all states whose names contain a space (e.g., "New York", "North Carolina"). Print the count and the list of names. Hint: `state_series.index` is an Index object, and an Index supports the `.str` accessor directly, just like a Series.

---
Now that we've covered Series, the next notebook introduces DataFrames, which are collections of Series.

<a href="16_5_Pandas_Dataframes.ipynb">Continue to: Dataframes →</a>